In [1]:
!pip install uv
!uv pip install -r https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/refs/heads/main/requirements.txt --system

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 65.4 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 133 packages in 809ms                                       
Prepared 8 packages in 1.34s                                             
Uninstalled 1 package in 12ms
Installed 8 packages in 177ms                               
 + async-lru==2.3.0
 + jedi==0.20.0
 + json5==0.15.0
 + jupyter-builder==1.1.0
 + jupyter-lsp==2.3.1
 - jupyter-server==2.18.2
 + jupyter-server==2.20.0
 + jupyterlab==4.6.1
 + jupyterlab-server==2.28.0


In [4]:
import torch
import torch.nn as nn

# Set seed for exact reproducibility of outputs
torch.manual_seed(42)

# Configurations
batch_size = 1      # b: We are processing a single sentence ("cats eat fish")
num_tokens = 3      # num_tokens: Three words in our sequence
d_in = 4            # d_in: Input embedding dimension per word
d_out = 4           # d_out: Output embedding dimension per word
num_heads = 2       # num_heads: We will split the work across 2 parallel heads
head_dim = d_out // num_heads  # head_dim: Each head manages 2 dimensions (4 // 2)

# Input Tensor x representing ["cats", "eat", "fish"]
x = torch.randn(batch_size, num_tokens, d_in)

print("--- INPUT TENSOR (x) ---")
print("Shape:", x.shape)
print(x)

--- INPUT TENSOR (x) ---
Shape: torch.Size([1, 3, 4])
tensor([[[ 0.3367,  0.1288,  0.2345,  0.2303],
         [-1.1229, -0.1863,  2.2082, -0.6380],
         [ 0.4617,  0.2674,  0.5349,  0.8094]]])


In [5]:
# Trainable weight matrices to project our input into Q, K, and V spaces
W_query = nn.Linear(d_in, d_out, bias=False)
W_key = nn.Linear(d_in, d_out, bias=False)
W_value = nn.Linear(d_in, d_out, bias=False)

# Final output projection layer to mix head contexts back together
out_proj = nn.Linear(d_out, d_out, bias=False)

# Create the upper triangular causal mask
mask = torch.triu(torch.ones(num_tokens, num_tokens), diagonal=1)

print("Causal Mask (Upper Triangular of 1s):\n", mask)

Causal Mask (Upper Triangular of 1s):
 tensor([[0., 1., 1.],
        [0., 0., 1.],
        [0., 0., 0.]])


In [6]:
# Linear projections
queries = W_query(x)
keys = W_key(x)
values = W_value(x)

print("Queries Shape:", queries.shape)
print("Keys Shape   :", keys.shape)
print("Values Shape :", values.shape)
print("\nGenerated Queries Tensor:\n", queries)

Queries Shape: torch.Size([1, 3, 4])
Keys Shape   : torch.Size([1, 3, 4])
Values Shape : torch.Size([1, 3, 4])

Generated Queries Tensor:
 tensor([[[-0.2649, -0.0397,  0.1738,  0.0548],
         [ 0.3662,  1.3071, -1.0046,  0.0585],
         [-0.5627, -0.2125,  0.3637,  0.0456]]], grad_fn=<UnsafeViewBackward0>)


In [7]:
# Split d_out (4) into num_heads (2) and head_dim (2)
queries = queries.view(batch_size, num_tokens, num_heads, head_dim)
keys = keys.view(batch_size, num_tokens, num_heads, head_dim)
values = values.view(batch_size, num_tokens, num_heads, head_dim)

print("After View Reshaping (batch, tokens, heads, head_dim):")
print("Queries Shape:", queries.shape)
print("\nQueries Tensor:\n", queries)

After View Reshaping (batch, tokens, heads, head_dim):
Queries Shape: torch.Size([1, 3, 2, 2])

Queries Tensor:
 tensor([[[[-0.2649, -0.0397],
          [ 0.1738,  0.0548]],

         [[ 0.3662,  1.3071],
          [-1.0046,  0.0585]],

         [[-0.5627, -0.2125],
          [ 0.3637,  0.0456]]]], grad_fn=<ViewBackward0>)


In [ ]:
# Swap dimension 1 (num_tokens) and dimension 2 (num_heads)
queries = queries.transpose(1, 2)
keys = keys.transpose(1, 2)
values = values.transpose(1, 2)

print("After Transposition (batch, heads, tokens, head_dim):")
print("Queries Shape:", queries.shape)
print("Keys Shape   :", keys.shape)
print("\nQueries Tensor (Head 0 is now separated from Head 1 for each of the word in other words the heads for queries are seperated so later they can be multiplied with key in a respective fashion):\n", queries)

After Transposition (batch, heads, tokens, head_dim):
Queries Shape: torch.Size([1, 2, 3, 2])
Keys Shape   : torch.Size([1, 2, 3, 2])

Queries Tensor (Head 0 is now separated from Head 1):
 tensor([[[[-0.2649, -0.0397],
          [ 0.3662,  1.3071],
          [-0.5627, -0.2125]],

         [[ 0.1738,  0.0548],
          [-1.0046,  0.0585],
          [ 0.3637,  0.0456]]]], grad_fn=<TransposeBackward0>)


In [13]:
print(f"queries: {queries}")
print(f"Keys: {keys}")

queries: tensor([[[[-0.2649, -0.0397],
          [ 0.3662,  1.3071],
          [-0.5627, -0.2125]],

         [[ 0.1738,  0.0548],
          [-1.0046,  0.0585],
          [ 0.3637,  0.0456]]]], grad_fn=<TransposeBackward0>)
Keys: tensor([[[[ 0.1799,  0.1574],
          [ 0.0543,  0.2965],
          [ 0.4996,  0.3018]],

         [[-0.1143,  0.0052],
          [-1.1980,  0.5401],
          [-0.3854,  0.1773]]]], grad_fn=<TransposeBackward0>)


In [14]:
keys_T = keys.transpose(2, 3)

In [ ]:
print(f"queries: {queries}")
print(f"KeysT: {keys_T}")

queries: tensor([[[[-0.2649, -0.0397],
          [ 0.3662,  1.3071],
          [-0.5627, -0.2125]],

         [[ 0.1738,  0.0548],
          [-1.0046,  0.0585],
          [ 0.3637,  0.0456]]]], grad_fn=<TransposeBackward0>)
Keyst: tensor([[[[ 0.1799,  0.0543,  0.4996],
          [ 0.1574,  0.2965,  0.3018]],

         [[-0.1143, -1.1980, -0.3854],
          [ 0.0052,  0.5401,  0.1773]]]], grad_fn=<TransposeBackward0>)


In [16]:
# Perform batched matrix multiplication
attn_scores = queries @ keys_T  # Shape becomes: (1, 2, 3, 3)

print("Attention Scores Shape (batch, heads, tokens, tokens):", attn_scores.shape)
print("\nRaw Attention Scores for Head 0:\n", attn_scores[0, 0])
print("\nRaw Attention Scores for Head 1:\n", attn_scores[0, 1])

Attention Scores Shape (batch, heads, tokens, tokens): torch.Size([1, 2, 3, 3])

Raw Attention Scores for Head 0:
 tensor([[-0.0539, -0.0261, -0.1443],
        [ 0.2717,  0.4074,  0.5774],
        [-0.1347, -0.0935, -0.3453]], grad_fn=<SelectBackward0>)

Raw Attention Scores for Head 1:
 tensor([[-0.0196, -0.1786, -0.0573],
        [ 0.1151,  1.2351,  0.3975],
        [-0.0413, -0.4111, -0.1321]], grad_fn=<SelectBackward0>)


In [17]:
# Convert our mask to booleans
mask_bool = mask.bool()

# Fill future attention score slots with -infinity
attn_scores.masked_fill_(mask_bool, -torch.inf)

print("Causal Masked Attention Scores for Head 0:\n", attn_scores[0, 0])

Causal Masked Attention Scores for Head 0:
 tensor([[-0.0539,    -inf,    -inf],
        [ 0.2717,  0.4074,    -inf],
        [-0.1347, -0.0935, -0.3453]], grad_fn=<SelectBackward0>)


In [18]:
# Compute keys.shape[-1] which is head_dim (2)
d_k = keys.shape[-1]

# Scale and Softmax
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

print("Attention Weights (Probabilities) for Head 0:\n", attn_weights[0, 0])
print("\nRow sums (all must equal 1.0):\n", attn_weights[0, 0].sum(dim=-1))

Attention Weights (Probabilities) for Head 0:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.4760, 0.5240, 0.0000],
        [0.3459, 0.3561, 0.2980]], grad_fn=<SelectBackward0>)

Row sums (all must equal 1.0):
 tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [19]:
context_vec = attn_weights @ values

print("Context Vectors Shape (batch, heads, tokens, head_dim):", context_vec.shape)
print("\nContext Vectors for Head 0:\n", context_vec[0, 0])

Context Vectors Shape (batch, heads, tokens, head_dim): torch.Size([1, 2, 3, 2])

Context Vectors for Head 0:
 tensor([[ 0.0620, -0.0369],
        [ 0.5089,  0.2332],
        [ 0.3450,  0.1178]], grad_fn=<SelectBackward0>)


In [20]:
# 1. Swap dimension 1 and 2 back: Shape (1, 2, 3, 2) -> (1, 3, 2, 2)
context_vec = context_vec.transpose(1, 2)

# 2. Make contiguous in memory and flatten the last two dimensions: (1, 3, 2, 2) -> (1, 3, 4)
context_vec = context_vec.contiguous().view(batch_size, num_tokens, d_out)

print("Merged Context Vector Shape:", context_vec.shape)
print("\nMerged Context Vectors for our Sentence:\n", context_vec[0])

Merged Context Vector Shape: torch.Size([1, 3, 4])

Merged Context Vectors for our Sentence:
 tensor([[ 0.0620, -0.0369,  0.0671,  0.0050],
        [ 0.5089,  0.2332,  0.6763,  0.8806],
        [ 0.3450,  0.1178,  0.3041,  0.4197]], grad_fn=<SelectBackward0>)


In [21]:
final_output = out_proj(context_vec)

print("--- FINAL OUTPUT TENSOR (Shape: 1, 3, 4) ---")
print("Shape:", final_output.shape)
print(final_output)

--- FINAL OUTPUT TENSOR (Shape: 1, 3, 4) ---
Shape: torch.Size([1, 3, 4])
tensor([[[ 0.0098, -0.0142,  0.0133,  0.0405],
         [-0.5313,  0.3630, -0.2094, -0.0219],
         [-0.1977,  0.1367, -0.1334,  0.0146]]], grad_fn=<UnsafeViewBackward0>)
